# 0. Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting'):
    os.chdir('..')

In [2]:
import pandas as pd 
import numpy as np

In [270]:
from scipy.optimize import minimize, OptimizeResult
from statsmodels.discrete.discrete_model import Logit

In [262]:
from core.dgp import *
from core.estimators import BaseEstimator

# 1. Group Targeting

In [ ]:
class DGP1:
    """ 
    Data Generating Process 1:
    \tau_i \in \{a_1, \dots, a_K\}  # K groups, each with a different treatment effect a_k 
    Y_i = \tau_i T_i + \epsilon_i, \epsilon_i \sim N(0, \sigma^2)  # outcome model
    T_i \sim Bernoulli(p)  # treatment assignment model
    """
    def __init__(
        self, num_groups: int, te_diff: float, group_func: callable, 
        noise_std: float = 1.0, treatment_assign_prob: float = 0.5, **kwargs
    ):
        """  
        DGP for model 1

        Params:
        -------
        num_groups: int, number of groups
        te_diff: float, treatment effect difference
        noise_std: float, standard deviation of the noise in the outcome model
        treatment_assign_prob: float, probability of treatment assignment
        """
        # attributes
        self.n_groups = num_groups
        self.te_diff = te_diff
        self.te_list = [1 + i * te_diff for i in range(num_groups)]
        self.group_func = group_func
        self.noise_std = noise_std
        self.treatment_assign_prob = treatment_assign_prob

    def generate_training_data(self, sample_size: int, seed=None) -> tuple:
        """
        Generate training data

        Params:
        -------
        sample_size: int, number of samples to generate

        Returns:
        -------
        tuple: 
            - X: np.ndarray, covariate
            - T: np.ndarray, treatment assignment
            - Y: np.ndarray, outcome
        """
        # set seed
        seed = seed if seed else np.random.randint(0, 1e6)
        np.random.seed(seed)

        # treatment effect function: given a covariate x, return the treatment effect of the group
        te_func = lambda x: self.te_list[self.group_func(x)]
        
        # generate data
        X = self.__generate_individual_characteristics(sample_size, seed)  # (sample_size, )
        T = np.random.binomial(1, 0.5, sample_size)  # (sample_size, )
        te_arr = np.array([te_func(x) for x in X])  # (sample_size, )
        Y = te_arr * T  + np.random.normal(0, self.noise_std, sample_size)  # (sample_size, )
        
        return X.reshape(-1, 1), T.reshape(-1, 1), Y.reshape(-1, 1)  # (sample_size, 1)
    
    def generate_testing_data(self, sample_size: int, seed=None) -> np.ndarray:
        seed = seed if seed else np.random.randint(0, 1e6)
        
        # generate data
        X = self.__generate_individual_characteristics(sample_size, seed) # (sample_size, )

        return X  # (sample_size, )

    def __generate_individual_characteristics(self, sample_size: int, seed: int) -> np.ndarray:
        np.random.seed(seed)
        return np.random.uniform(-3, 3, sample_size)

# 2. Personalized Pricing

Based on (Dube and Misra 2023, Journal of Political Economy)

In [252]:
class PersonalizedPricingDGP(object):
    def __init__(self, cov_dim: int = 1):
        self.cov_dim = cov_dim
        self.util_const_map = np.random.uniform(0, 1, size=(cov_dim, 1))  # (cov_dim, 1)
        self.util_price_map = np.random.uniform(-0.1, 0, size=(cov_dim, 1))  # (cov_dim, 1)

    def generate_training_data(
        self, sample_size: int, price_lb: float, price_ub: float, price_diff: float, seed=None
    ) -> tuple:
        """
        Generate training data

        Params:
        -------
        sample_size: int, number of samples to generate
        price_lb: float, lower bound of the price
        price_ub: float, upper bound of the price
        price_diff: float, price difference
        seed: int, random seed

        Returns:
        -------
        tuple: 
            - X: np.ndarray, covariate
            - T: np.ndarray, treatment assignment
            - Y: np.ndarray, outcome
        """
        # generate data
        # individual characteristics
        X_arr = self.__generate_individual_characteristics(sample_size)  # (sample_size, cov_dim)
        
        # price (treaments)
        price_arr = np.random.choice(np.arange(price_lb, price_ub, price_diff), sample_size).reshape(-1, 1)  # (sample_size, 1)

        # error
        err_arr = np.random.gumbel(loc=0, scale=1, size=(sample_size, 1))  # (sample_size, 1)

        # utility: (sample_size, 1)
        util_arr = X_arr @ self.util_const_map + X_arr @ self.util_price_map * price_arr + err_arr

        # demand: (sample_size, 1), buy if utility > 0, else don't buy.
        demand_arr = (util_arr > 0).astype(int)

        return X_arr, price_arr, demand_arr
    
    def generate_testing_data(self, sample_size: int) -> np.ndarray:        
        return self.__generate_individual_characteristics(sample_size)  # (sample_size, )

    def __generate_individual_characteristics(self, sample_size: int) -> np.ndarray:
        return np.random.normal(loc=3, scale=0.5, size=(sample_size, self.cov_dim))

In [253]:
dgp = PersonalizedPricingDGP(cov_dim=1)

In [271]:
cov_arr, price_arr, demand_arr = dgp.generate_training_data(
    sample_size=1000, price_lb=1, price_ub=10, price_diff=1
)

In [275]:
class PricingPlugIn(BaseEstimator):
    def __init__(self, cov_dim: int):
        # attributes
        self.cov_dim = cov_dim

        # placeholders
        self.util_const_map = np.zeros((self.cov_dim, 1))  # (cov_dim, 1)
        self.util_price_map = np.zeros((self.cov_dim, 1))  # (cov_dim, 1)
        self.opt_result = None

    def fit(self, covariates: np.ndarray, prices: np.ndarray, outcomes: np.ndarray):
        # fit the utility function
        exog = np.concatenate([covariates, prices * covariates], axis=1)
        model = Logit(outcomes, exog)
        result = model.fit()

        self.util_const_map = result.params[:self.cov_dim].reshape(-1, 1)  # (cov_dim, 1)
        self.util_price_map = result.params[self.cov_dim:].reshape(-1, 1)  # (cov_dim, 1)

        return self

    def purchase_prob(self, covariates: np.ndarray, price: float) -> np.ndarray:
        """ 
        Params:
        -------
        covariates: np.ndarray, (n_obs, cov_dim)
        price: float, price
        """
        logit = np.exp(covariates @ self.util_const_map + (price * covariates) @ self.util_price_map)
        return logit / (1 + logit)
    
    def objective_func(self, covariates: np.ndarray, price: float, delta: float = 0.99) -> float:
        purchase_prob = self.purchase_prob(covariates, price)
        return np.mean(price * purchase_prob / (1 - delta * purchase_prob))
    
    def optimize(self, covariates: np.ndarray, delta: float = 0.99) -> OptimizeResult:
        """ 
        
        Params: 
        -------
        covariates: np.ndarray, (n_obs, cov_dim)
        delta: float, discount factor

        Returns:
        -------
        opt_result: scipy.optimize.OptimizeResult
        """
        self.opt_result = minimize(
            lambda x: -self.objective_func(covariates, x, delta), x0=1, method='L-BFGS-B',
        )
        return self.opt_result
    
    def estimate_targeting_value(self, covariates: np.ndarray, delta: float = 0.99) -> float:
        """ 
        Estimate the targeting value

        Params:
        -------
        covariates: np.ndarray, (n_obs, cov_dim)
        delta: float, discount factor

        Returns:
        -------
        targeting_value: float
        """
        if self.opt_result is None:
            _ = self.optimize(covariates, delta)

        return - self.opt_result.fun
    
    def get_targeting_policy(self, covariates: np.ndarray, delta: float = 0.99) -> float:
        """ 
        Get the targeting policy

        Params:
        -------
        covariates: np.ndarray, (n_obs, cov_dim)
        delta: float, discount factor

        Returns:
        -------
        targeting_policy: np.ndarray, (n_obs, )
        """
        if self.opt_result is None:
            _ = self.optimize(covariates, delta)
        
        return self.opt_result.x[0]